In [108]:
class Config:

    #Globals
     #Globals
    batch_size = 16
    num_classes = 3  # classes, seizure/no seizure
    epochs = 25   # Epoch iterations
    time_step_length = 5
    row_hidden = 128  # hidden neurons in conv layers
    col_hidden = 128   # hidden neurons in the Bi-LSTM layers
    RANDOM_SEED = 3333    
    N_TIME_STEPS = 125   # 50 records in each sequence
    N_FEATURES = 3     # mag,hr,roi_Ratio,output
    step = 100           # window overlap = 50 -10 = 40  (80% overlap)
    N_CLASSES = 3      # class label
    learning_rate = 0.0000001
    k = 5 # number of k folds
    target_class_count=58250

In [180]:
import pandas as pd
import numpy as np
import json

class OsdbDataLabelGenerator:
    def __init__(self, file_path, sampling_rate=25):
        self.file_path = file_path  # Path to the JSON file
        self.sampling_rate = sampling_rate  # Sampling rate (Hz)
        self.df_sensordata = None  # To store the processed DataFrame

    def load_data(self):
        """Load and flatten the JSON data into a DataFrame."""
        with open(self.file_path, 'r') as file:
            raw_json = json.load(file)
        
        flattened_data = []
        for attribute in raw_json:
            user_id = attribute.get('userId', None)
            datapoints = attribute.get('datapoints', [])
            
            for point in datapoints:
                event_id = point.get('eventId', None)
                hr = point.get('hr', [])
                rawData = point.get('rawData', [])
                flattened_data.append({
                    'eventId': event_id,
                    'userId': user_id,
                    'hr': hr,
                    'rawData': rawData,
                })
        
        # Convert to DataFrame
        self.df_sensordata = pd.DataFrame(flattened_data)
        
        # Add a sequential 'Id' column, starting from 1
        self.df_sensordata['Id'] = range(1, len(self.df_sensordata) + 1)

    def calculate_fft(self, raw_data):
        """Calculate FFT for the raw data."""
        raw_data = raw_data - np.mean(raw_data)  # Remove the DC component
        fft_result = np.fft.fft(raw_data)  # Compute FFT
        frequencies = np.fft.fftfreq(len(raw_data), d=1/self.sampling_rate)  # Compute frequencies
        fft_magnitude = np.abs(fft_result)  # Compute the magnitude
        positive_frequencies = frequencies[:len(frequencies)//2]  # Only positive frequencies
        positive_fft_magnitude = fft_magnitude[:len(frequencies)//2]  # Only positive FFT magnitudes
        return positive_frequencies, positive_fft_magnitude

    def add_fft_column(self):
        """Add an FFT column to the DataFrame with zero-padding to ensure each entry has 125 values."""
        fft_results = []
        for _, row in self.df_sensordata.iterrows():
            raw_data = np.array(row['rawData'])
            _, positive_fft_magnitude = self.calculate_fft(raw_data)  # Calculate FFT for the row
            # Apply zero padding to ensure the FFT column has exactly 125 values
            padded_fft = np.pad(positive_fft_magnitude, (0, 125 - len(positive_fft_magnitude)), 'constant', constant_values=0)
            fft_results.append(list(padded_fft))  # Append padded FFT result
        self.df_sensordata['FFT'] = fft_results

    def process_data(self):
        """Process the data through all stages and return the final DataFrame."""
        # Step 1: Load the data
        self.load_data()

        # Step 2: Add FFT column with padding
        self.add_fft_column()

        return self.df_sensordata


In [ ]:
# Example usage
data_path = '../Data/osdb_3min_allSeizures.json'  # Path to main dataset
labels_path = '../Data/labels_expanded.csv'       # Path to labels dataset

# Initialize the data processor
processor = OsdbDataLabelGenerator(data_path, sampling_rate=25)

# Process the data
df_result_with_labels = processor.process_data()

# Display the final merged DataFrame
df_result_with_labels.head(50)

array([  407,   764,  4924,  5483,  5486,  5610,  5705,  6897,  7775,
        7772,  8800, 11587, 11591, 15923, 17219,  1046,  5031,  5087,
        5254,  5288,  5580,  5635,  5637,  5721,  5745,  5891,  5889,
        6476,  6587,  6590,  6668,  6717,  6732,  6761,  6767,  6808,
        6815,  6840,  6847,  6884,  6886,  6998,  7006,  7007,  7036,
        7044,  7125,  7126,  7219,  7222,  7258,  7262,  7365,  7363,
        7434,  7823,  8726,  8420,  8738,  8875,  8960,  8970,  8998,
        9005,  9401,  9470,  9475, 12206, 12214, 14157, 14159, 15230,
        5596,  5595,  7357,  8661,  9627, 12618, 12624, 12629, 15039,
        9828, 12763, 12973, 14101, 15208, 15417, 21458, 21561, 21569,
       21695, 21603, 21797, 21855, 21865, 21866, 21867, 21886, 24380,
       26071, 26077, 26988, 26992, 27272, 27786, 28725, 28734,   119,
       31339, 31397, 31404, 31420, 31421, 34756, 34759,   115, 31402,
       36812, 36799, 36872, 40784, 40913, 41062, 42147, 44115, 44137,
       42626, 45209,

In [189]:
def assign_timesteps(group):
    """Assign timestep column within each eventId group."""
    group = group.reset_index(drop=True)  # Reset index for proper slicing
    group['timestep'] = (group.index // 125) + 1  # Calculate timestep
    return group

labels_path = '../Data/binary_osd_labels.csv'       # Path to labels dataset

df = pd.read_csv(labels_path)

# Apply the function to assign timestep for each eventId group
df = (df.groupby('eventId', group_keys=False)
    .apply(assign_timesteps)
)

# Print the resulting DataFrame to verify
print(df.head(50))  # Adjust to print more rows if needed


       Id  eventId  label  timestep
0   95001      115      1         1
1   95002      115      1         1
2   95003      115      1         1
3   95004      115      1         1
4   95005      115      1         1
5   95006      115      1         1
6   95007      115      1         1
7   95008      115      1         1
8   95009      115      1         1
9   95010      115      1         1
10  95011      115      1         1
11  95012      115      1         1
12  95013      115      1         1
13  95014      115      1         1
14  95015      115      1         1
15  95016      115      1         1
16  95017      115      1         1
17  95018      115      1         1
18  95019      115      1         1
19  95020      115      1         1
20  95021      115      1         1
21  95022      115      1         1
22  95023      115      1         1
23  95024      115      1         1
24  95025      115      1         1
25  95026      115      1         1
26  95027      115      1   

In [207]:
# Print every 125th row
every_125th_row = df.iloc[::125]
print("Every 125th row in the DataFrame:")
len(df_result_with_labels)
#csv_path = '../Data/labels_new.csv'
#every_125th_row.to_csv(csv_path, index=False)


Every 125th row in the DataFrame:


3949

In [211]:
data_path = '../Data/binary_osd_labels.csv'  # Path to main dataset
df = pd.read_csv(data_path)
len(df)/125

3181.0

In [195]:
import pandas as pd
import numpy as np
import json

class OsdbDataLabelGenerator:
    def __init__(self, file_path, sampling_rate=25, labels_path=None):
        """
        Initialize the generator.
        
        Args:
            file_path (str): Path to the JSON file containing the main dataset.
            sampling_rate (int): Sampling rate for FFT calculations (Hz).
            labels_path (str, optional): Path to the CSV file containing the labels. Default is None.
        """
        self.file_path = file_path  # Path to the JSON file
        self.sampling_rate = sampling_rate  # Sampling rate (Hz)
        self.labels_path = labels_path  # Path to the labels CSV file
        self.df_sensordata = None  # To store the processed DataFrame
        self.labels_df = pd.read_csv(labels_path) if labels_path else None

    def load_data(self):
        """Load and flatten the JSON data into a DataFrame."""
        with open(self.file_path, 'r') as file:
            raw_json = json.load(file)
        
        flattened_data = []
        for attribute in raw_json:
            user_id = attribute.get('userId', None)
            datapoints = attribute.get('datapoints', [])
            
            for point in datapoints:
                event_id = point.get('eventId', None)
                hr = point.get('hr', [])
                rawData = point.get('rawData', [])
                flattened_data.append({
                    'eventId': event_id,
                    'userId': user_id,
                    'hr': hr,
                    'rawData': rawData,
                })
        
        # Convert to DataFrame
        self.df_sensordata = pd.DataFrame(flattened_data)
        
        # Add a sequential 'Id' column, starting from 1
        self.df_sensordata['Id'] = range(1, len(self.df_sensordata) + 1)

    def calculate_fft(self, raw_data):
        """Calculate FFT for the raw data."""
        raw_data = raw_data - np.mean(raw_data)  # Remove the DC component
        fft_result = np.fft.fft(raw_data)  # Compute FFT
        frequencies = np.fft.fftfreq(len(raw_data), d=1/self.sampling_rate)  # Compute frequencies
        fft_magnitude = np.abs(fft_result)  # Compute the magnitude
        positive_frequencies = frequencies[:len(frequencies)//2]  # Only positive frequencies
        positive_fft_magnitude = fft_magnitude[:len(frequencies)//2]  # Only positive FFT magnitudes
        return positive_frequencies, positive_fft_magnitude

    def add_fft_column(self):
        """Add an FFT column to the DataFrame with zero-padding to ensure each entry has 125 values."""
        fft_results = []
        for _, row in self.df_sensordata.iterrows():
            raw_data = np.array(row['rawData'])
            _, positive_fft_magnitude = self.calculate_fft(raw_data)  # Calculate FFT for the row
            # Apply zero padding to ensure the FFT column has exactly 125 values
            padded_fft = np.pad(positive_fft_magnitude, (0, 125 - len(positive_fft_magnitude)), 'constant', constant_values=0)
            fft_results.append(list(padded_fft))  # Append padded FFT result
        self.df_sensordata['FFT'] = fft_results

    def filter_and_merge_labels(self):
        """Filter and merge the labels DataFrame with the processed sensor data."""
        if self.labels_df is not None:
            # Step 1: Filter df_sensordata to retain only rows with eventId present in labels_df
            filtered_df_result = self.df_sensordata[self.df_sensordata['eventId'].isin(self.labels_df['eventId'])]

            # Step 2: Ensure the correct length before merging
            print("Length of filtered_df_result before merge:", len(filtered_df_result))
            print("Length of labels_df before merge:", len(self.labels_df))

            # Sort filtered_df_result to match the order of eventId in labels_df
            #filtered_df_result = filtered_df_result.set_index('eventId')

            # Sort labels_df to ensure matching eventId order
            labels_df = self.labels_df.set_index('eventId')

            # Step 3: Check if lengths match after filtering
            if len(filtered_df_result) == len(labels_df):
                # Step 4: Add the Label column from labels_df to filtered_df_result
                filtered_df_result['Label'] = labels_df['Label'].values
                self.df_sensordata = filtered_df_result.reset_index()
            else:
                print("Error: The lengths of the filtered DataFrame and labels DataFrame do not match after filtering. Cannot merge.")

    def process_data(self):
        """Process the data through all stages and return the final DataFrame."""
        # Step 1: Load the data
        self.load_data()

        # Step 2: Add FFT column with padding
        self.add_fft_column()

        # Step 3: Filter and merge with labels if provided
        self.filter_and_merge_labels()

        return self.df_sensordata

# Example usage
data_path = '../Data/osdb_3min_allSeizures.json'  # Path to main dataset
labels_path = '../Data/labels_new.csv'       # Path to labels dataset

# Initialize the data processor
processor = OsdbDataLabelGenerator(data_path, sampling_rate=25, labels_path=labels_path)

# Process the data
df_result_with_labels = processor.process_data()

# Display the final merged DataFrame
df_result_with_labels.head(50)


Length of filtered_df_result before merge: 3172
Length of labels_df before merge: 3181
Error: The lengths of the filtered DataFrame and labels DataFrame do not match after filtering. Cannot merge.


,eventId,userId,hr,rawData,Id,FFT
0,407,39,67,"[1496, 1480, 1500, 1492, 1496, 1484, 1500, 149...",1,"[1.2960299500264227e-11, 143.05125737182817, 5..."
1,407,39,67,"[1492, 1508, 1496, 1476, 1484, 1476, 1496, 150...",2,"[9.094947017729282e-13, 75.02350794818989, 31...."
2,407,39,68,"[1488, 1496, 1484, 1492, 1492, 1508, 1504, 148...",3,"[2.2737367544323206e-13, 91.25440903139302, 81..."
3,407,39,69,"[1488, 1476, 1480, 1504, 1496, 1508, 1484, 148...",4,"[1.3642420526593924e-11, 101.37768172754971, 7..."
4,407,39,69,"[1504, 1488, 1504, 1492, 1484, 1500, 1496, 149...",5,"[7.275957614183426e-12, 116.42740204040989, 77..."
5,407,39,70,"[1500, 1480, 1496, 1488, 1480, 1488, 1484, 149...",6,"[1.4097167877480388e-11, 64.1735150114216, 146..."
6,407,39,72,"[1500, 1476, 1484, 1468, 1500, 1472, 1496, 148...",7,"[1.8189894035458565e-12, 2033.4679618695818, 2..."
7,407,39,72,"[1432, 1304, 1320, 1180, 1308, 1392, 1384, 129...",8,"[1.318767317570746e-11, 8830.29196306564, 2535..."
8,407,39,71,"[1504, 1476, 1448, 1592, 1612, 1420, 1348, 166...",9,"[1.0686562745831907e-11, 2977.6280564474355, 6..."
9,407,39,71,"[1532, 1532, 1488, 1516, 1512, 1508, 1536, 156...",10,"[3.410605131648481e-12, 10345.864896884996, 99..."


In [179]:
df = pd.read_csv("../Data/binary_osd_labels.csv")
# Print all eventIds in the DataFrame
print("All eventIds in the DataFrame:")
print(df['eventId'].tolist())

# Print unique eventIds in the DataFrame
unique_event_ids = df['eventId'].unique()
unique_event_ids


All eventIds in the DataFrame:
[5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 5635, 56

array([ 5635,  5637,  6668,  8726,  8738, 15923,  7219, 28725,  7222,
       15417, 21561,  6717, 21569,  5705,  6732,  5721,  7258,  7772,
        7262,  7775,  8800,  9828, 41062,  6761, 44137,  6767,  5745,
         115,   119,  5254,  7823,  6808, 45209,  6815, 42147,  5288,
       12973,  9401, 31420,  7357, 31421, 15039, 21695,  7365, 45781,
        6884,  6897,  9470,  8960,  5889,  5891,  9475,  7434,  8970,
       21797,  8998,  9005,  4924, 24380, 11587, 11591, 12618,  6476,
       14157, 14159, 12624, 40784, 45393,  6998,  7006,  7007, 15208,
       21865, 21866,  5483, 21867, 26988,  5486, 26992,  7036, 15230,
       21886,  7044,   407, 47000, 47002,  9627, 53665, 53666,  5031,
       12206, 12214,  6587,  6590, 36799, 34756, 34759,  5580, 36812,
       40913, 21458,  7125,  7126,  8661, 26071,  5595,  5596, 12763,
       26077,  5087,  5610,  6847], dtype=int64)

In [ ]:
df = pd.read_csv("../Data/binary_osd_labels.csv")
# Print all eventIds in the DataFrame
print("All eventIds in the DataFrame:")
print(df['eventId'].tolist())

# Print unique eventIds in the DataFrame
unique_event_ids = df['eventId'].unique()
unique_event_ids


In [ ]:
import pandas as pd
import numpy as np
import json

class OsdbDataLabelGenerator:
    def __init__(self, file_path, sampling_rate=25, labels_path=None):
        self.file_path = file_path  # Path to the JSON file
        self.sampling_rate = sampling_rate  # Sampling rate (Hz)
        self.labels_path = labels_path  # Path to the labels CSV file
        self.df_sensordata = None  # To store the processed DataFrame
        self.labels_df = pd.read_csv(labels_path) if labels_path else None

    def load_data(self):
        """Load and flatten the JSON data into a DataFrame."""
        with open(self.file_path, 'r') as file:
            raw_json = json.load(file)
        
        flattened_data = []
        for attribute in raw_json:
            user_id = attribute.get('userId', None)
            datapoints = attribute.get('datapoints', [])
            for point in datapoints:
                event_id = point.get('eventId', None)
                hr = point.get('hr', [])
                rawData = point.get('rawData', [])
                flattened_data.append({
                    'eventId': event_id,
                    'userId': user_id,
                    'hr': hr,
                    'rawData': rawData,
                })
        
        self.df_sensordata = pd.DataFrame(flattened_data)
        self.df_sensordata['Id'] = range(1, len(self.df_sensordata) + 1)

    def calculate_fft(self, raw_data):
        """Calculate FFT for the raw data."""
        raw_data = raw_data - np.mean(raw_data)
        fft_result = np.fft.fft(raw_data)
        frequencies = np.fft.fftfreq(len(raw_data), d=1/self.sampling_rate)
        fft_magnitude = np.abs(fft_result)
        positive_frequencies = frequencies[:len(frequencies)//2]
        positive_fft_magnitude = fft_magnitude[:len(frequencies)//2]
        return positive_frequencies, positive_fft_magnitude

    def add_fft_column(self):
        """Add an FFT column to the DataFrame with zero-padding to ensure each entry has 125 values."""
        fft_results = []
        for _, row in self.df_sensordata.iterrows():
            raw_data = np.array(row['rawData'])
            _, positive_fft_magnitude = self.calculate_fft(raw_data)
            padded_fft = np.pad(positive_fft_magnitude, (0, 125 - len(positive_fft_magnitude)), 'constant', constant_values=0)
            fft_results.append(list(padded_fft))
        self.df_sensordata['FFT'] = fft_results

    def filter_and_merge_labels(self):
        """Filter, merge, and ensure proper grouping and ordering."""
        if self.labels_df is not None:
            # Step 1: Filter df_sensordata to retain only rows with eventId present in labels_df
            filtered_df_result = self.df_sensordata[self.df_sensordata['eventId'].isin(self.labels_df['eventId'])]

            # Step 2: Sort filtered_df_result by eventId and reset index
            filtered_df_result = filtered_df_result.sort_values(by="eventId").reset_index(drop=True)

            # Step 3: Sort labels_df by eventId and reset index
            sorted_labels_df = self.labels_df.sort_values(by="eventId").reset_index(drop=True)

            # Step 4: Match the labels to the filtered_df_result
            if len(filtered_df_result) == len(sorted_labels_df):
                filtered_df_result['Label'] = sorted_labels_df['Label'].values
            else:
                print("Error: Length mismatch between filtered sensor data and labels. Cannot merge.")
                return

            # Step 5: Sort the final DataFrame by eventId and Id
            filtered_df_result = filtered_df_result.sort_values(by=['eventId', 'Id']).reset_index(drop=True)

            # Update the processed DataFrame
            self.df_sensordata = filtered_df_result


    def process_data(self):
        self.load_data()
        self.add_fft_column()
        self.filter_and_merge_labels()
        return self.df_sensordata

# Example usage
data_path = '../Data/osdb_3min_allSeizures.json'  # Path to main dataset
labels_path = '../Data/labels_expanded.csv'       # Path to labels dataset

processor = OsdbDataLabelGenerator(data_path, sampling_rate=25, labels_path=labels_path)
df_result_with_labels = processor.process_data()

df_result_with_labels.head(50)


,eventId,userId,hr,rawData,Id,FFT,Label
0,115,39,-1,"[1066.988281, 1007.777771, 1025.935669, 1019.7...",3252,"[5.434230843093246e-11, 107.70246469996025, 31...",0
1,115,39,-1,"[1060.196167, 1083.032837, 1105.137085, 1082.7...",3253,"[3.296918293926865e-12, 174.17424007818258, 69...",0
2,115,39,89,"[1074.519409, 983.991882, 953.989502, 1036.247...",3254,"[2.3874235921539366e-12, 816.8217155063495, 41...",0
3,115,39,87,"[926.792297, 989.359375, 1108.866089, 1027.314...",3255,"[2.3533175408374518e-11, 2603.862033277181, 77...",1
4,115,39,90,"[1116.207886, 1075.405029, 1037.496948, 1057.5...",3256,"[1.318767317570746e-11, 10201.320394248973, 33...",0
5,115,39,93,"[1028.334595, 879.463501, 697.756409, 760.2210...",3257,"[2.3078428057488054e-11, 11934.515902025589, 3...",1
6,115,39,103,"[1102.731201, 981.142212, 962.613098, 1192.107...",3258,"[1.3301360013429075e-11, 2806.7458218705265, 1...",1
7,115,39,93,"[1028.334595, 879.463501, 697.756409, 760.2210...",3259,"[2.3078428057488054e-11, 11934.515902025589, 3...",1
8,115,39,104,"[968.561829, 1140.448975, 802.216919, 1156.546...",3260,"[2.660272002685815e-11, 100.05588139546211, 47...",1
9,115,39,114,"[1467.694824, 1212.633545, 851.192078, 408.411...",3261,"[2.9103830456733704e-11, 985.984674552243, 286...",1


In [ ]:
import pandas as pd
import numpy as np
import json

class OsdbDataLabelGenerator:
    def __init__(self, file_path, sampling_rate=25, labels_path=None):
        self.file_path = file_path  # Path to the JSON file
        self.sampling_rate = sampling_rate  # Sampling rate (Hz)
        self.labels_path = labels_path  # Path to the labels CSV file
        self.df_sensordata = None  # To store the processed DataFrame
        self.labels_df = pd.read_csv(labels_path) if labels_path else None

    def load_data(self):
        """Load and flatten the JSON data into a DataFrame."""
        with open(self.file_path, 'r') as file:
            raw_json = json.load(file)
        
        flattened_data = []
        for attribute in raw_json:
            user_id = attribute.get('userId', None)
            datapoints = attribute.get('datapoints', [])
            for point in datapoints:
                event_id = point.get('eventId', None)
                hr = point.get('hr', [])
                rawData = point.get('rawData', [])
                flattened_data.append({
                    'eventId': event_id,
                    'userId': user_id,
                    'hr': hr,
                    'rawData': rawData,
                })
        
        self.df_sensordata = pd.DataFrame(flattened_data)
        self.df_sensordata['Id'] = range(1, len(self.df_sensordata) + 1)

    def calculate_fft(self, raw_data):
        """Calculate FFT for the raw data."""
        raw_data = raw_data - np.mean(raw_data)
        fft_result = np.fft.fft(raw_data)
        frequencies = np.fft.fftfreq(len(raw_data), d=1/self.sampling_rate)
        fft_magnitude = np.abs(fft_result)
        positive_frequencies = frequencies[:len(frequencies)//2]
        positive_fft_magnitude = fft_magnitude[:len(frequencies)//2]
        return positive_frequencies, positive_fft_magnitude

    def add_fft_column(self):
        """Add an FFT column to the DataFrame with zero-padding to ensure each entry has 125 values."""
        fft_results = []
        for _, row in self.df_sensordata.iterrows():
            raw_data = np.array(row['rawData'])
            _, positive_fft_magnitude = self.calculate_fft(raw_data)
            padded_fft = np.pad(positive_fft_magnitude, (0, 125 - len(positive_fft_magnitude)), 'constant', constant_values=0)
            fft_results.append(list(padded_fft))
        self.df_sensordata['FFT'] = fft_results

    def filter_and_merge_labels(self):
        """Filter, merge, and ensure proper grouping and ordering."""
        if self.labels_df is not None:
            # Step 1: Filter df_sensordata to retain only rows with eventId present in labels_df
            filtered_df_result = self.df_sensordata[self.df_sensordata['eventId'].isin(self.labels_df['eventId'])]

            # Step 2: Sort filtered_df_result by eventId and reset index
            filtered_df_result = filtered_df_result.sort_values(by="eventId").reset_index(drop=True)

            # Step 3: Sort labels_df by eventId and reset index
            sorted_labels_df = self.labels_df.sort_values(by="eventId").reset_index(drop=True)

            # Step 4: Match the labels to the filtered_df_result
            if len(filtered_df_result) == len(sorted_labels_df):
                filtered_df_result['Label'] = sorted_labels_df['Label'].values
            else:
                print("Error: Length mismatch between filtered sensor data and labels. Cannot merge.")
                return

            # Step 5: Sort the final DataFrame by eventId and Id
            filtered_df_result = filtered_df_result.sort_values(by=['eventId', 'Id']).reset_index(drop=True)

            # Update the processed DataFrame
            self.df_sensordata = filtered_df_result


    def process_data(self):
        self.load_data()
        self.add_fft_column()
        self.filter_and_merge_labels()
        return self.df_sensordata

# Example usage
data_path = '../Data/osdb_3min_allSeizures.json'  # Path to main dataset
labels_path = '../Data/labels_expanded.csv'       # Path to labels dataset

processor = OsdbDataLabelGenerator(data_path, sampling_rate=25, labels_path=labels_path)
df_result_with_labels = processor.process_data()

df_result_with_labels.head(50)


,eventId,userId,hr,rawData,Id,FFT,Label
0,115,39,-1,"[1066.988281, 1007.777771, 1025.935669, 1019.7...",3252,"[5.434230843093246e-11, 107.70246469996025, 31...",0
1,115,39,-1,"[1060.196167, 1083.032837, 1105.137085, 1082.7...",3253,"[3.296918293926865e-12, 174.17424007818258, 69...",0
2,115,39,89,"[1074.519409, 983.991882, 953.989502, 1036.247...",3254,"[2.3874235921539366e-12, 816.8217155063495, 41...",0
3,115,39,87,"[926.792297, 989.359375, 1108.866089, 1027.314...",3255,"[2.3533175408374518e-11, 2603.862033277181, 77...",1
4,115,39,90,"[1116.207886, 1075.405029, 1037.496948, 1057.5...",3256,"[1.318767317570746e-11, 10201.320394248973, 33...",0
5,115,39,93,"[1028.334595, 879.463501, 697.756409, 760.2210...",3257,"[2.3078428057488054e-11, 11934.515902025589, 3...",1
6,115,39,103,"[1102.731201, 981.142212, 962.613098, 1192.107...",3258,"[1.3301360013429075e-11, 2806.7458218705265, 1...",1
7,115,39,93,"[1028.334595, 879.463501, 697.756409, 760.2210...",3259,"[2.3078428057488054e-11, 11934.515902025589, 3...",1
8,115,39,104,"[968.561829, 1140.448975, 802.216919, 1156.546...",3260,"[2.660272002685815e-11, 100.05588139546211, 47...",1
9,115,39,114,"[1467.694824, 1212.633545, 851.192078, 408.411...",3261,"[2.9103830456733704e-11, 985.984674552243, 286...",1


In [175]:
import pandas as pd
import numpy as np
import json

class OsdbDataLabelGenerator:
    def __init__(self, file_path, sampling_rate=25, labels_path=None):
        self.file_path = file_path  # Path to the JSON file
        self.sampling_rate = sampling_rate  # Sampling rate (Hz)
        self.labels_path = labels_path  # Path to the labels CSV file
        self.df_sensordata = None  # To store the processed DataFrame
        self.labels_df = pd.read_csv(labels_path) if labels_path else None

    def load_data(self):
        """Load and flatten the JSON data into a DataFrame."""
        with open(self.file_path, 'r') as file:
            raw_json = json.load(file)
        
        flattened_data = []
        for attribute in raw_json:
            user_id = attribute.get('userId', None)
            datapoints = attribute.get('datapoints', [])
            for point in datapoints:
                event_id = point.get('eventId', None)
                hr = point.get('hr', [])
                rawData = point.get('rawData', [])
                flattened_data.append({
                    'eventId': event_id,
                    'userId': user_id,
                    'hr': hr,
                    'rawData': rawData,
                })
        
        self.df_sensordata = pd.DataFrame(flattened_data)
        self.df_sensordata['Id'] = range(1, len(self.df_sensordata) + 1)

    def calculate_fft(self, raw_data):
        """Calculate FFT for the raw data."""
        raw_data = raw_data - np.mean(raw_data)
        fft_result = np.fft.fft(raw_data)
        frequencies = np.fft.fftfreq(len(raw_data), d=1/self.sampling_rate)
        fft_magnitude = np.abs(fft_result)
        positive_frequencies = frequencies[:len(frequencies)//2]
        positive_fft_magnitude = fft_magnitude[:len(frequencies)//2]
        return positive_frequencies, positive_fft_magnitude

    def add_fft_column(self):
        """Add an FFT column to the DataFrame with zero-padding to ensure each entry has 125 values."""
        fft_results = []
        for _, row in self.df_sensordata.iterrows():
            raw_data = np.array(row['rawData'])
            _, positive_fft_magnitude = self.calculate_fft(raw_data)
            padded_fft = np.pad(positive_fft_magnitude, (0, 125 - len(positive_fft_magnitude)), 'constant', constant_values=0)
            fft_results.append(list(padded_fft))
        self.df_sensordata['FFT'] = fft_results

    def filter_and_merge_labels(self):
        """Filter, merge, and ensure proper grouping and ordering."""
        if self.labels_df is not None:
            # Step 1: Filter df_sensordata to retain only rows with eventId present in labels_df
            filtered_df_result = self.df_sensordata[self.df_sensordata['eventId'].isin(self.labels_df['eventId'])]

            # Step 2: Sort filtered_df_result by eventId and reset index
            filtered_df_result = filtered_df_result.sort_values(by="eventId").reset_index(drop=True)

            # Step 3: Sort labels_df by eventId and reset index
            sorted_labels_df = self.labels_df.sort_values(by="eventId").reset_index(drop=True)

            # Step 4: Match the labels to the filtered_df_result
            if len(filtered_df_result) == len(sorted_labels_df):
                filtered_df_result['Label'] = sorted_labels_df['Label'].values
            else:
                print("Error: Length mismatch between filtered sensor data and labels. Cannot merge.")
                return

            # Step 5: Sort the final DataFrame by eventId and Id
            filtered_df_result = filtered_df_result.sort_values(by=['eventId', 'Id']).reset_index(drop=True)

            # Update the processed DataFrame
            self.df_sensordata = filtered_df_result


    def process_data(self):
        self.load_data()
        self.add_fft_column()
        self.filter_and_merge_labels()
        return self.df_sensordata

# Example usage
data_path = '../Data/osdb_3min_allSeizures.json'  # Path to main dataset
labels_path = '../Data/labels_expanded.csv'       # Path to labels dataset

processor = OsdbDataLabelGenerator(data_path, sampling_rate=25, labels_path=labels_path)
df_result_with_labels = processor.process_data()

df_result_with_labels.head(50)


,eventId,userId,hr,rawData,Id,FFT,Label
0,115,39,-1,"[1066.988281, 1007.777771, 1025.935669, 1019.7...",3252,"[5.434230843093246e-11, 107.70246469996025, 31...",0
1,115,39,-1,"[1060.196167, 1083.032837, 1105.137085, 1082.7...",3253,"[3.296918293926865e-12, 174.17424007818258, 69...",0
2,115,39,89,"[1074.519409, 983.991882, 953.989502, 1036.247...",3254,"[2.3874235921539366e-12, 816.8217155063495, 41...",0
3,115,39,87,"[926.792297, 989.359375, 1108.866089, 1027.314...",3255,"[2.3533175408374518e-11, 2603.862033277181, 77...",1
4,115,39,90,"[1116.207886, 1075.405029, 1037.496948, 1057.5...",3256,"[1.318767317570746e-11, 10201.320394248973, 33...",0
5,115,39,93,"[1028.334595, 879.463501, 697.756409, 760.2210...",3257,"[2.3078428057488054e-11, 11934.515902025589, 3...",1
6,115,39,103,"[1102.731201, 981.142212, 962.613098, 1192.107...",3258,"[1.3301360013429075e-11, 2806.7458218705265, 1...",1
7,115,39,93,"[1028.334595, 879.463501, 697.756409, 760.2210...",3259,"[2.3078428057488054e-11, 11934.515902025589, 3...",1
8,115,39,104,"[968.561829, 1140.448975, 802.216919, 1156.546...",3260,"[2.660272002685815e-11, 100.05588139546211, 47...",1
9,115,39,114,"[1467.694824, 1212.633545, 851.192078, 408.411...",3261,"[2.9103830456733704e-11, 985.984674552243, 286...",1


In [89]:
import pandas as pd
import os

class IpdDataReshaper:
    def __init__(self, dataframe):
        if dataframe is None or not isinstance(dataframe, pd.DataFrame) or dataframe.empty:
            raise ValueError("Invalid DataFrame provided to IpdDataReshaper.")
        
        required_columns = ['eventId', 'userId', 'hr', 'rawData', 'FFT', 'labels']
        missing_columns = [col for col in required_columns if col not in dataframe.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
        
        self.df = dataframe

    def reshape_data(self):
        reshaped_rows = []
        
        for idx, row in self.df.iterrows():
            event_id = row['eventId']
            Id = row['Id']
            user_id = row['userId']
            hr = row['hr']
            rawData = row['rawData']
            fft = row['FFT']
            labels = row['labels']

            # Ensure rawData and FFT are list-like
            if not isinstance(rawData, list) or not isinstance(fft, list):
                raise ValueError(f"Invalid data format at index {idx}: 'rawData' and 'FFT' must be lists.")

            # Transpose rawData and FFT
            rawData_transposed = rawData[:125]
            fft_transposed = fft[:125]

            # Replicate information
            repeated_info = {
                'eventId': [event_id] * 125,
                'Id': [Id] * 125,
                'userId': [user_id] * 125,
                'hr': [hr] * 125,
                'labels': [labels] * 125,
            }

            # Create reshaped rows
            for i in range(125):
                reshaped_rows.append({
                    'eventId': repeated_info['eventId'][i],
                    'Id': repeated_info['Id'][i],
                    'userId': repeated_info['userId'][i],
                    'hr': repeated_info['hr'][i],
                    'rawData': rawData_transposed[i],
                    'FFT': fft_transposed[i],
                    'labels': repeated_info['labels'][i],

                })
        
        return pd.DataFrame(reshaped_rows)


In [90]:
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline

class OsdbInterpolator:
    def __init__(self, df, column_to_interpolate):
        """
        Initialize the Interpolator class with a DataFrame and the column to interpolate.
        """
        self.df = df
        self.column_to_interpolate = column_to_interpolate

    def interpolate_column(self, new_column_name='interpolated_hr', interval=125, time_step=5):
        """
        Interpolate the specified column using the provided logic.
        
        Parameters:
        - new_column_name: Name of the new column to store interpolated values.
        - interval: Interval to sample the original column (e.g., every 125th element).
        - time_step: Time step in seconds for the interpolation process.
        """
        # Step 1: Extract every nth element from the specified column
        original_values = self.df[self.column_to_interpolate]
        selected_elements = original_values[0::interval]
        x = np.array(selected_elements)

        # Step 2: Create an array representing the time (in `time_step` intervals)
        time_values = np.arange(len(x)) * time_step

        # Step 3: Create a CubicSpline object for interpolation
        cs = CubicSpline(time_values, x, bc_type='clamped')

        # Step 4: Generate new time values for finer granularity
        num_original_points = len(x)
        new_time_values = np.linspace(0, (num_original_points - 1) * time_step, num_original_points * interval)

        # Step 5: Generate interpolated values
        interpolated_values = cs(new_time_values)

        # Step 6: Add the interpolated values to the DataFrame
        self.df[new_column_name] = interpolated_values[:len(self.df)]  # Match the original DataFrame length

        # Step 7: Rearrange columns so that 'label' is always last
        columns = list(self.df.columns)
        if 'label' in columns:
            columns.remove('label')
            columns.append('label')
        self.df = self.df[columns]


    def get_dataframe(self):
        """
        Return the updated DataFrame with interpolated values.
        """
        return self.df

In [98]:
data_path = '../Data/osdb_3min_allSeizures.json'  # Replace with your JSON file path
labels_df = pd.read_csv('../Data/labels_aggregated.csv')

processor = OsdbDataLabelGenerator(data_path, Config.N_TIME_STEPS)
df_result = processor.process_data()
df_result


,eventId,userId,hr,rawData,Id,FFT
0,407,39,67,"[1496, 1480, 1500, 1492, 1496, 1484, 1500, 149...",1,"[1.2960299500264227e-11, 143.05125737182817, 5..."
1,407,39,67,"[1492, 1508, 1496, 1476, 1484, 1476, 1496, 150...",2,"[9.094947017729282e-13, 75.02350794818989, 31...."
2,407,39,68,"[1488, 1496, 1484, 1492, 1492, 1508, 1504, 148...",3,"[2.2737367544323206e-13, 91.25440903139302, 81..."
3,407,39,69,"[1488, 1476, 1480, 1504, 1496, 1508, 1484, 148...",4,"[1.3642420526593924e-11, 101.37768172754971, 7..."
4,407,39,69,"[1504, 1488, 1504, 1492, 1484, 1500, 1496, 149...",5,"[7.275957614183426e-12, 116.42740204040989, 77..."
...,...,...,...,...,...,...
3944,53666,39,-1,"[1001.399048, 1004.223083, 1015.488037, 994.85...",3945,"[8.640199666842818e-12, 73.47271094056933, 72...."
3945,53666,39,-1,"[982.698303, 987.676086, 1013.998047, 1001.670...",3946,"[1.3301360013429075e-11, 128.37305265198444, 4..."
3946,53666,39,-1,"[999.175659, 1009.316589, 1002.189575, 988.712...",3947,"[2.0463630789890885e-12, 95.03426164728164, 88..."
3947,53666,39,-1,"[1009.403809, 999.952026, 1012.031616, 999.607...",3948,"[1.1482370609883219e-11, 140.2430826956883, 73..."


In [84]:
data_path = '../Data/osdb_3min_allSeizures.json'  # Replace with your JSON file path
labels_df = pd.read_csv('../Data/labels_aggregated.csv')

processor = OsdbDataLabelGenerator(data_path, Config.N_TIME_STEPS, labels_df)
df_result = processor.process_data()
df_result

,eventId,userId,hr,rawData,Id,labels,FFT
0,407,39,67,"[1496, 1480, 1500, 1492, 1496, 1484, 1500, 149...",1,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.2960299500264227e-11, 143.05125737182817, 5..."
1,407,39,67,"[1492, 1508, 1496, 1476, 1484, 1476, 1496, 150...",2,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[9.094947017729282e-13, 75.02350794818989, 31...."
2,407,39,68,"[1488, 1496, 1484, 1492, 1492, 1508, 1504, 148...",3,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2.2737367544323206e-13, 91.25440903139302, 81..."
3,407,39,69,"[1488, 1476, 1480, 1504, 1496, 1508, 1484, 148...",4,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.3642420526593924e-11, 101.37768172754971, 7..."
4,407,39,69,"[1504, 1488, 1504, 1492, 1484, 1500, 1496, 149...",5,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[7.275957614183426e-12, 116.42740204040989, 77..."
...,...,...,...,...,...,...,...
3944,53666,39,-1,"[1001.399048, 1004.223083, 1015.488037, 994.85...",3945,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, ...","[8.640199666842818e-12, 73.47271094056933, 72...."
3945,53666,39,-1,"[982.698303, 987.676086, 1013.998047, 1001.670...",3946,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, ...","[1.3301360013429075e-11, 128.37305265198444, 4..."
3946,53666,39,-1,"[999.175659, 1009.316589, 1002.189575, 988.712...",3947,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, ...","[2.0463630789890885e-12, 95.03426164728164, 88..."
3947,53666,39,-1,"[1009.403809, 999.952026, 1012.031616, 999.607...",3948,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, ...","[1.1482370609883219e-11, 140.2430826956883, 73..."


In [77]:
import pandas as pd
import numpy as np
import json

# Config placeholder
class Config:
    N_TIME_STEPS = 125
    time_step_length = 1  # Placeholder value; replace with your actual time step length

# OsdbDataLabelGenerator Class
class OsdbDataLabelGenerator:
    def __init__(self, file_path, sampling_rate, labels_df):
        self.file_path = file_path
        self.sampling_rate = sampling_rate
        self.labels_df = labels_df
        self.df_sensordata = None

    def load_data(self):
        """Load and flatten the JSON data into a DataFrame."""
        with open(self.file_path, 'r') as file:
            raw_json = json.load(file)
        
        flattened_data = []
        for attribute in raw_json:
            user_id = attribute.get('userId', None)
            datapoints = attribute.get('datapoints', [])
            
            for point in datapoints:
                event_id = point.get('eventId', None)
                hr = point.get('hr', [])
                rawData = point.get('rawData', [])
                flattened_data.append({
                    'eventId': event_id,
                    'userId': user_id,
                    'hr': hr,
                    'rawData': rawData,
                })
        
        self.df_sensordata = pd.DataFrame(flattened_data)
        self.df_sensordata['Id'] = range(1, len(self.df_sensordata) + 1)

    def add_labels(self):
        """Merge the labels into the DataFrame based on eventId."""
        if self.labels_df is not None:
            self.df_sensordata = self.df_sensordata.merge(
                self.labels_df[['eventId', 'labels']], on='eventId', how='left'
            )

    def calculate_fft(self, raw_data):
        """Calculate FFT for the raw data."""
        raw_data = raw_data - np.mean(raw_data)
        fft_result = np.fft.fft(raw_data)
        frequencies = np.fft.fftfreq(len(raw_data), d=1 / self.sampling_rate)
        fft_magnitude = np.abs(fft_result)
        return frequencies[:len(frequencies) // 2], fft_magnitude[:len(frequencies) // 2]

    def add_fft_column(self):
        """Add FFT data to the DataFrame."""
        fft_results = []
        for _, row in self.df_sensordata.iterrows():
            raw_data = np.array(row['rawData'])
            _, fft_magnitude = self.calculate_fft(raw_data)
            padded_fft = np.pad(fft_magnitude, (0, 125 - len(fft_magnitude)), 'constant', constant_values=0)
            fft_results.append(list(padded_fft))
        self.df_sensordata['FFT'] = fft_results

    def process_data(self):
        """Execute all processing steps."""
        self.load_data()
        self.add_labels()
        self.add_fft_column()
        return self.df_sensordata

# IpdDataReshaper Class
class IpdDataReshaper:
    def __init__(self, dataframe):
        if dataframe is None or not isinstance(dataframe, pd.DataFrame) or dataframe.empty:
            raise ValueError("Invalid DataFrame provided to IpdDataReshaper.")
        required_columns = ['eventId', 'userId', 'hr', 'rawData', 'FFT', 'labels']
        missing_columns = [col for col in required_columns if col not in dataframe.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
        self.df = dataframe

    def reshape_data(self):
        reshaped_rows = []
        for idx, row in self.df.iterrows():
            event_id = row['eventId']
            Id = row['Id']
            user_id = row['userId']
            hr = row['hr']
            rawData = row['rawData']
            fft = row['FFT']
            labels = row['labels']

            # Validate list-like inputs
            if not isinstance(rawData, list) or not isinstance(fft, list) or not isinstance(labels, list):
                raise ValueError(f"Invalid data format at index {idx}: All data fields must be lists.")

            # Repeat labels and validate
            expanded_labels = []
            for label in labels:
                expanded_labels.extend([label] * 125)

            if len(expanded_labels) != len(rawData):
                raise ValueError(
                    f"Length mismatch: labels ({len(expanded_labels)}) and rawData ({len(rawData)}) lengths differ."
                )

            repeated_info = {
                'eventId': [event_id] * len(rawData),
                'Id': [Id] * len(rawData),
                'userId': [user_id] * len(rawData),
                'hr': [hr] * len(rawData),
            }

            for i in range(len(rawData)):
                reshaped_rows.append({
                    'eventId': repeated_info['eventId'][i],
                    'Id': repeated_info['Id'][i],
                    'userId': repeated_info['userId'][i],
                    'hr': repeated_info['hr'][i],
                    'rawData': rawData[i],
                    'FFT': fft[i],
                    'labels': expanded_labels[i],
                })
        return pd.DataFrame(reshaped_rows)

# Main Processing Script
data_path = '../Data/osdb_3min_allSeizures.json'  # Replace with your JSON file path
labels_df = pd.read_csv('../Data/labels_aggregated.csv')

processor = OsdbDataLabelGenerator(data_path, Config.N_TIME_STEPS, labels_df)
df_result = processor.process_data()

reshaper = IpdDataReshaper(df_result)
reshaped_df = reshaper.reshape_data()
reshaped_df.head()

# Interpolation placeholder
class OsdbInterpolator:
    def __init__(self, dataframe, column_to_interpolate):
        self.df = dataframe
        self.column_to_interpolate = column_to_interpolate

    def interpolate_column(self, new_column_name, interval, time_step):
        self.df[new_column_name] = self.df[self.column_to_interpolate]  # Replace with actual interpolation logic

    def get_dataframe(self):
        return self.df

interpolator = OsdbInterpolator(reshaped_df, column_to_interpolate="hr")
interpolator.interpolate_column(new_column_name="interpolated_hr", interval=Config.N_TIME_STEPS, time_step=Config.time_step_length)
df_sensor_data = interpolator.get_dataframe()
df_sensor_data.head()


ValueError: Invalid data format at index 0: All data fields must be lists.

In [19]:
# Filter df2 to include only rows with eventId in df1
ordered_event_ids = labels['eventId'].unique()
filtered_df2 = df_sensor_data[df_sensor_data['eventId'].isin(ordered_event_ids)]

In [20]:
labels["label"].value_counts()
#label
#0.0    3250
#Name: count, dtype: int64

label
1    196755
0    183995
Name: count, dtype: int64

In [28]:
import pandas as pd

# Assuming you already have the 'labels' DataFrame
# Step 1: Reduce every 125th row per eventId
labels_reduced = (
    labels.groupby('eventId')
    .apply(lambda x: x.iloc[::125])  # Take every 125th row
    .reset_index(drop=True)
)

# Step 2: Aggregate labels into a list per eventId
labels_aggregated = (
    labels_reduced.groupby('eventId')['label']
    .apply(list)  # Collect all labels as a list
    .reset_index()
)

# Rename for clarity
labels_aggregated.rename(columns={'label': 'Label_List'}, inplace=True)

# Check the number of values saved for eventId 407
event_407_labels = labels_aggregated[labels_aggregated['eventId'] == 407]['Label_List'].iloc[0]

# Save to CSV
csv_path = '../Data/labels_aggregated.csv'
labels_aggregated.to_csv(csv_path, index=False)

# Output the result and path
print("Labels for eventId 407:", event_407_labels)
print("CSV saved at:", csv_path)


Labels for eventId 407: [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
CSV saved at: ../Data/labels_aggregated.csv


In [95]:
import pandas as pd

# Assuming you already have the 'labels_aggregated' DataFrame

# Step 1: Explode the 'Label_List' column to create separate rows
labels_expanded = labels_aggregated.explode('Label_List').reset_index(drop=True)

# Step 2: Rename the columns for clarity (optional)
labels_expanded.rename(columns={'Label_List': 'Label'}, inplace=True)

# Step 3: Check the result (optional)
print(labels_expanded.head())

# Step 4: Save the result to a new CSV
csv_path_expanded = '../Data/labels_expanded.csv'
labels_expanded.to_csv(csv_path_expanded, index=False)

# Output the result and path
print("Expanded DataFrame saved at:", csv_path_expanded)

   eventId Label
0      115     1
1      115     1
2      115     1
3      115     1
4      115     1
Expanded DataFrame saved at: ../Data/labels_expanded.csv
